# Wind Forecast Analysis - January 2024

This notebook analyzes forecasted and actual wind power generation in the UK. 

**Objectives:**
1. Understand forecast error characteristics.
2. Analyze historical wind generation to estimate reliable MW capacity.
3. Document assumptions, trade-offs, and reasoning for recommendations.
4. Include only forecasts published 4–48 hours before target generation to match the app logic.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

## Step 1: Load Data

In [ ]:
# Load CSVs
df_actual = pd.read_csv('wind_actuals_jan2024.csv', parse_dates=['startTime'])
df_forecast = pd.read_csv('wind_forecasts_jan2024.csv', parse_dates=['startTime','publishTime'])

display(df_actual.head())
display(df_forecast.head())

## Step 2: Merge Forecasts and Actuals

In [ ]:
df = pd.merge(df_forecast, df_actual, on='startTime', suffixes=('_forecast','_actual'))
df = df.dropna(subset=['generation_forecast','generation_actual'])
display(df.head())

## Step 3: Compute Forecast Horizon & Filter 4–48 Hours

In [ ]:
# Forecast horizon in hours
df['forecast_horizon_hours'] = (df['startTime'] - df['publishTime']).dt.total_seconds() / 3600

# Keep only forecasts published 4–48 hours before target generation
df = df[(df['forecast_horizon_hours'] >= 4) & (df['forecast_horizon_hours'] <= 48)]

print(f"Number of records after filtering by 4-48 hour horizon: {len(df)}")

## Step 4: Compute Errors

In [ ]:
df['error'] = df['generation_forecast'] - df['generation_actual']
df['abs_error'] = df['error'].abs()
df['rel_error'] = df['abs_error'] / df['generation_actual'] * 100
df['hour'] = df['startTime'].dt.hour
df.head()

## Step 5: Error Statistics

In [ ]:
print('Mean Absolute Error (MW):', df['abs_error'].mean())
print('Median Absolute Error (MW):', df['abs_error'].median())
print('P99 Absolute Error (MW):', df['abs_error'].quantile(0.99))
print('Mean Relative Error (%):', df['rel_error'].mean())

## Step 6: Error vs Forecast Horizon

In [ ]:
plt.figure(figsize=(10,5))
sns.scatterplot(x='forecast_horizon_hours', y='abs_error', data=df, alpha=0.5)
plt.xlabel('Forecast Horizon (hours)')
plt.ylabel('Absolute Error (MW)')
plt.title('Forecast Error vs Forecast Horizon')
plt.show()

## Step 7: Error by Hour of Day

In [ ]:
hourly_error = df.groupby('hour')['abs_error'].mean()
plt.figure(figsize=(10,5))
sns.barplot(x=hourly_error.index, y=hourly_error.values, palette='viridis')
plt.xlabel('Hour of Day')
plt.ylabel('Mean Absolute Error (MW)')
plt.title('Forecast Error by Hour of Day')
plt.show()

## Step 8: Plot Actual vs Forecast

In [ ]:
plt.figure(figsize=(15,5))
plt.plot(df['startTime'], df['generation_actual'], label='Actual', color='blue')
plt.plot(df['startTime'], df['generation_forecast'], label='Forecast', color='green')
plt.xlabel('Time')
plt.ylabel('Wind Generation (MW)')
plt.title('Actual vs Forecast Wind Generation - Jan 2024')
plt.legend()
plt.show()

## Step 9: Estimate Reliable Wind Generation

In [ ]:
mean_gen = df['generation_actual'].mean()
std_gen = df['generation_actual'].std()
reliable_gen = mean_gen - std_gen
print(f'Mean actual generation: {mean_gen:.2f} MW')
print(f'Std deviation: {std_gen:.2f} MW')
print(f'Approx. reliable wind generation: {reliable_gen:.2f} MW')

## Step 10: Save Merged Data (Optional)

In [ ]:
df.to_csv('wind_analysis_ready_filtered.csv', index=False)
print('Filtered merged CSV saved.')

### Notes / Assumptions:
- Only forecasts with horizon between 4–48 hours are considered.
- Absolute and relative errors are computed using filtered data.
- Reliable wind generation = mean - 1*std (conservative estimate).
- Plots and metrics now match the forecast monitoring app logic.